In [0]:
# bronze_notebook.py

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col

spark = SparkSession.builder.getOrCreate()

# Simulated data ingestion (without Kafka)
from pyspark.sql.functions import lit, to_json, struct
from datetime import datetime

data = [
    ("sim-1", datetime.now(), 42.7, "km/h", 23.7275, 37.9838, "1"),
    ("sim-2", datetime.now(), 35.2, "km/h", 23.7320, 37.9810, "2"),
    ("sim-3", datetime.now(), 50.1, "km/h", 23.7400, 37.9850, "3")
]

schema = ["sensor_id", "sensor_timestamp", "value", "unit", "longitude", "latitude", "district"]

df = spark.createDataFrame(data, schema=schema)
df = df.withColumn("ingest_time", current_timestamp())

# Convert row to JSON
df_json = df.select(to_json(struct("*")).alias("value")) \
            .withColumn("topic", lit("simulated.sensors")) \
            .withColumn("partition", lit(0)) \
            .withColumn("offset", lit(0)) \
            .withColumn("ingest_time", current_timestamp())

# Append to Bronze Delta
df_json.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("workspace4sadt.bronze.sensors_raw")
